# SB-vs-BB RAISE pass — headless GPU

Adds a **Raise** option to the **SB-vs-BB** (blind-vs-blind) single-raised pot: re-solves all 17 flop boards with `--raise-x 3` so every *facing-a-bet* spot gains Fold/**Call**/**Raise** (currently SB-vs-BB vs-bet spots are Fold/Call only). First-to-act & checked-to spots stay Check/Bet.

Same 2-part flow as the BTN raise pass. **Per commit:** set `PART='A'` (boards 0–8), then `'B'` (9–16), Save & Run All each time (GPU T4 x2, Internet On). Download `records_raise_sb_A.json` and `records_raise_sb_B.json` and send both back — they build/sign into a raise-enabled `sb_vs_bb` pack locally.

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On) -- pulls SB-vs-BB + 17 boards.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.presets import SCENARIOS, BOARDS
print('scenario:', SCENARIOS['sb_vs_bb_srp']['label'], '| boards:', len(BOARDS), '| raise-to: 3x the bet')

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

## Solve one part, then Save & Run All. Recommended: **2 parts** (`PART='A'` then `'B'`).

Set `SPLIT=2` and run `PART='A'` (boards 0-8), then `PART='B'` (9-16) in a second commit
— each ~2-4.5 h, safely under Kaggle's ~9 h limit. If a part ever runs long, drop to
`SPLIT=3`, or split its `ROOTS` by hand (e.g. `'0,1,2'` then `'3,4,5'`). The merge is
board-wise, so any grouping recombines cleanly.

In [ ]:
# --- Pick a split. Given ~1 h per 6 boards for Check/Bet, the raise tree is bigger
# but a 2-part split (~2-4.5 h each) stays well under Kaggle's ~9 h commit limit. ---
SPLIT = 2        # 2 = two commits (recommended); 3 = three shorter commits
PART  = 'A'      # 2-part: 'A' (boards 0-8), 'B' (9-16)
                 # 3-part: 'A' (0-5), 'B' (6-11), 'C' (12-16)
assert SPLIT in (2, 3), 'SPLIT must be 2 or 3'
assert PART in {2: ('A', 'B'), 3: ('A', 'B', 'C')}[SPLIT], f'PART {PART!r} is not valid for SPLIT={SPLIT}'
ROOTS = ({2: {'A': '0,1,2,3,4,5,6,7,8', 'B': '9,10,11,12,13,14,15,16'},
          3: {'A': '0,1,2,3,4,5', 'B': '6,7,8,9,10,11', 'C': '12,13,14,15,16'}}[SPLIT])[PART]
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
print(f'SPLIT={SPLIT} PART={PART} -> boards {ROOTS}')
subprocess.run(
    ['python', '-m', 'pokertrainer.content_yield',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--scenario', 'sb_vs_bb_srp', '--raise-x', '3', '--roots', ROOTS,
     '--out', f'/kaggle/working/cy_raise_sb_{PART}'],
    cwd='/kaggle/working/poker', env=env)

In [ ]:
# Expose records_raise_sb_<PART>.json for download (fails loudly if nothing completed).
import json, shutil, os
try:
    rep = json.load(open(f'/kaggle/working/cy_raise_sb_{PART}/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit(f'No cy_raise_sb_{PART} output -- check the cell above and cy_raise_sb_{PART}/boards/*.ERROR.txt')
done = rep.get('boards_completed') or []
print('PART', PART, '| boards completed:', done, '| missing:', rep.get('boards_missing'))
if not done:
    raise SystemExit(f'No boards completed in PART {PART} -- check cy_raise_sb_{PART}/boards/*.ERROR.txt')
recs = json.load(open(f'/kaggle/working/cy_raise_sb_{PART}/records.json'))
with_raise = sum(1 for r in recs if 'raise' in (r.get('actions') or list((r.get('ev') or {}).keys())))
shutil.copy(f'/kaggle/working/cy_raise_sb_{PART}/records.json', f'/kaggle/working/records_raise_sb_{PART}.json')
print('DOWNLOAD -> records_raise_sb_%s.json (%d KB, %d records, %d with a raise option)'
      % (PART, os.path.getsize(f'/kaggle/working/records_raise_sb_{PART}.json')//1024, len(recs), with_raise))